# SIMULACION DE INCENDIO CON QGIS

## Superposicion de mapas Elevacion, Bosques, Humedales

* Simulacion de seleccion de una zona del mapa

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import box  # solo para crear el cuadrado de análisis

# --------------------------
# 1. Cargar capas
# --------------------------
bosques = gpd.read_file("bosques.gpkg")
humedales = gpd.read_file("humedades.gpkg")
elevaciones = gpd.read_file("elevaciones.gpkg")

# --------------------------
# 2. Asegurar mismo CRS
# --------------------------
# Tomamos el CRS de 'bosques' como referencia si está disponible
target_crs = bosques.crs if bosques.crs is not None else (
    humedales.crs if humedales.crs is not None else elevaciones.crs
)

if humedales.crs != target_crs and humedales.crs is not None:
    humedales = humedales.to_crs(target_crs)
if elevaciones.crs != target_crs and elevaciones.crs is not None:
    elevaciones = elevaciones.to_crs(target_crs)

# --------------------------
# 3. Interponer las capas (solo eso)
# --------------------------
# Intersección polígono (bosque ∩ humedal). No se hace nada más con esto aquí.
bosque_humedal = gpd.overlay(bosques, humedales, how="intersection")

# --------------------------
# 4. Seleccionar un área de análisis (cuadrado)
# --------------------------
# Seleccionamos bounds desde la intersección si existe, sino desde las capas disponibles
if len(bosque_humedal) > 0:
    base_bounds = bosque_humedal.total_bounds
elif len(bosques) > 0:
    base_bounds = bosques.total_bounds
elif len(humedales) > 0:
    base_bounds = humedales.total_bounds
elif len(elevaciones) > 0:
    base_bounds = elevaciones.total_bounds
else:
    raise ValueError("Todas las capas están vacías: no hay datos para definir el área de análisis.")

minx, miny, maxx, maxy = base_bounds
width = maxx - minx
height = maxy - miny

# Definimos el lado del cuadrado como el 25% de la menor dimensión del extent (ajustable)
if width == 0 and height == 0:
    side = 1000.0
elif width == 0:
    side = height * 0.25
elif height == 0:
    side = width * 0.25
else:
    side = min(width, height) * 0.25

cx = (minx + maxx) / 2.0
cy = (miny + maxy) / 2.0
half = side / 2.0

analysis_square = box(cx - half, cy - half, cx + half, cy + half)
square_gdf = gpd.GeoDataFrame(geometry=[analysis_square], crs=target_crs)

# --------------------------
# 5. Recortar capas al cuadrado (para la visualización)
# --------------------------
# Polígonos
try:
    bosques_sel = gpd.clip(bosques, square_gdf)
except Exception:
    # fallback si clip no está disponible o falla
    bosques_sel = bosques[bosques.intersects(analysis_square)]

try:
    humedales_sel = gpd.clip(humedales, square_gdf)
except Exception:
    humedales_sel = humedales[humedales.intersects(analysis_square)]

# Elevaciones (puntos) — filtrado espacial simple
elevaciones_sel = elevaciones[elevaciones.intersects(analysis_square)]

# Intersección bosque∩humedal dentro del cuadrado (si quieres ver solo la parte que "afecta")
bosque_humedal_sel = bosque_humedal[bosque_humedal.intersects(analysis_square)]

# --------------------------
# 6. Mostrar mapa de la zona seleccionada
# --------------------------
fig, ax = plt.subplots(figsize=(10, 10))

# Borde del cuadrado
square_gdf.boundary.plot(ax=ax, linestyle="--", linewidth=1, alpha=0.8)

# Bosque ∩ Humedal (los bosques que afectan)
if not bosque_humedal_sel.empty:
    if "tipo_bosqu" in bosque_humedal_sel.columns:
        bosque_humedal_sel.plot(ax=ax, column="tipo_bosqu", cmap="Set3", alpha=0.6, legend=True)
    else:
        bosque_humedal_sel.plot(ax=ax, alpha=0.6, edgecolor="k")

# Humedales (si hay)
if not humedales_sel.empty:
    if "tipo" in humedales_sel.columns:
        humedales_sel.plot(ax=ax, column="tipo", cmap="Blues", alpha=0.5, legend=True)
    else:
        humedales_sel.plot(ax=ax, alpha=0.5, edgecolor="blue")

# Elevaciones (puntos)
if not elevaciones_sel.empty:
    if "altura" in elevaciones_sel.columns:
        elevaciones_sel.plot(ax=ax, column="altura", cmap="terrain", markersize=30, legend=True)
    else:
        elevaciones_sel.plot(ax=ax, markersize=30)

ax.set_title("Área de análisis (cuadrado) — Bosque∩Humedal y Elevaciones")
ax.set_xlim(cx - half, cx + half)
ax.set_ylim(cy - half, cy + half)
ax.set_xlabel("Coordenada X")
ax.set_ylabel("Coordenada Y")
plt.tight_layout()
plt.show()

# --------------------------
# Fin: aquí sólo se interpuso, se definió el cuadrado y se mostró la zona seleccionada.
# No hay más cálculos ni salidas impresas (listas, estadísticas, etc.).
# Después seguimos con los pasos siguientes cuando quieras.
# --------------------------


## SIMULACION DE EXPANSION DE FUEGO

* Expansion de fuego simple, a partir de la altura

In [ ]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import box
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from scipy.interpolate import griddata

# --------------------------
# 1. Cargar capas
# --------------------------
bosques = gpd.read_file("bosques.gpkg")
humedales = gpd.read_file("humedades.gpkg")
elevaciones = gpd.read_file("elevaciones.gpkg")

target_crs = bosques.crs
humedales = humedales.to_crs(target_crs)
elevaciones = elevaciones.to_crs(target_crs)

# --------------------------
# 2. Función para definir zona de análisis
# --------------------------
def definirzona(iniciox, inicioy, finx, finy):
    """Recorta las capas a un rectángulo definido por coordenadas."""
    zona = box(iniciox, inicioy, finx, finy)
    zona_gdf = gpd.GeoDataFrame(geometry=[zona], crs=target_crs)

    bosques_sel = gpd.clip(bosques, zona_gdf)
    humedales_sel = gpd.clip(humedales, zona_gdf)
    elevaciones_sel = gpd.clip(elevaciones, zona_gdf)

    return bosques_sel, humedales_sel, elevaciones_sel, zona_gdf

# --------------------------
# 3. Seleccionar zona de prueba
# --------------------------
bosques_sel, humedales_sel, elevaciones_sel, zona_gdf = definirzona(
    500000, 8000000, 502000, 8002000
)

# --------------------------
# 3b. Si la zona está vacía, elegir automáticamente una zona válida
# --------------------------
if elevaciones_sel.empty:
    print("Zona vacía, seleccionando automáticamente una zona válida...")

    minx, miny, maxx, maxy = elevaciones.total_bounds
    ancho = (maxx - minx) * 0.25   # 25% del ancho total
    alto  = (maxy - miny) * 0.25   # 25% del alto total

    iniciox = minx + (maxx - minx) * 0.25
    finx    = iniciox + ancho
    inicioy = miny + (maxy - miny) * 0.25
    finy    = inicioy + alto

    bosques_sel, humedales_sel, elevaciones_sel, zona_gdf = definirzona(
        iniciox, inicioy, finx, finy
    )
    print(f" Nueva zona definida: ({iniciox}, {inicioy}) a ({finx}, {finy})")

# --------------------------
# 4. Preparar grilla de elevaciones
# --------------------------
if elevaciones_sel.empty:
    raise ValueError("No hay puntos de elevación en la zona seleccionada. Prueba otros límites.")

coords = np.array([[geom.centroid.x, geom.centroid.y] for geom in elevaciones_sel.geometry],
                  dtype=np.float64)
z = elevaciones_sel["altura"].astype(float).to_numpy()

nx, ny = 60, 60
xi = np.linspace(coords[:, 0].min(), coords[:, 0].max(), nx)
yi = np.linspace(coords[:, 1].min(), coords[:, 1].max(), ny)
X, Y = np.meshgrid(xi, yi)
Z = griddata((coords[:, 0], coords[:, 1]), z, (X, Y), method="cubic")

# --------------------------
# 5. Simulación de fuego (versión base solo elevaciones)
# --------------------------
T = np.zeros_like(Z)

# foco inicial en el centro del área
cx, cy = nx // 2, ny // 2
T[cx-2:cx+3, cy-2:cy+3] = 1000  # ignición inicial

# parámetros de propagación
D = 0.4     # difusión
alpha = 0.01
Tign = 200  # temperatura de ignición
dt = 0.1

def laplacian(Z):
    return (np.roll(Z,1,0) + np.roll(Z,-1,0) + np.roll(Z,1,1) + np.roll(Z,-1,1) - 4*Z)

def get_Tem(T):
    S = (T > Tign).astype(float) * 30.0
    dT = D * laplacian(T) - alpha * T + S
    return T + dT * dt

# --------------------------
# 6. Visualización animada en 3D
# --------------------------
fig = plt.figure(figsize=(9,7))
ax = fig.add_subplot(111, projection="3d")

def animate(frame):
    global T
    T = get_Tem(T)
    ax.clear()
    ax.plot_surface(X, Y, Z, cmap="terrain", alpha=0.5, linewidth=0)
    Thorn = np.clip(T / 1000, 0, 1)
    ax.plot_surface(X, Y, Z + 10, facecolors=plt.cm.hot(Thorn), alpha=0.7)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Elevación")

ani = FuncAnimation(fig, animate, frames=120, interval=120, blit=False)
plt.show()


## SIMULACION  DE EXPANSION DE FUEGO PAR.2 -> AGREGADO DE BOSQUES Y HUMEDALES

* Agregado de bosques y humedales para la expansion del fuego


In [ ]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import box, Point
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from scipy.interpolate import griddata
import random


# --------------------------
# 1. Cargar capas
# --------------------------
bosques = gpd.read_file("bosques.gpkg")
humedales = gpd.read_file("humedades.gpkg")
elevaciones = gpd.read_file("elevaciones.gpkg")

target_crs = bosques.crs
humedales = humedales.to_crs(target_crs)
elevaciones = elevaciones.to_crs(target_crs)

# --------------------------
# 2. Función para definir zona de análisis
# --------------------------
def definirzona(iniciox, inicioy, finx, finy):
    """Recorta las capas a un rectángulo definido por coordenadas."""
    zona = box(iniciox, inicioy, finx, finy)
    zona_gdf = gpd.GeoDataFrame(geometry=[zona], crs=target_crs)

    bosques_sel = gpd.clip(bosques, zona_gdf)
    humedales_sel = gpd.clip(humedales, zona_gdf)
    elevaciones_sel = gpd.clip(elevaciones, zona_gdf)

    return bosques_sel, humedales_sel, elevaciones_sel, zona_gdf

# --------------------------
# 3. Seleccionar zona de prueba
# --------------------------
bosques_sel, humedales_sel, elevaciones_sel, zona_gdf = definirzona(
    500000, 8000000, 502000, 8002000
)

# --------------------------
# 3b. Si la zona está vacía, elegir automáticamente una zona válida
# --------------------------
if elevaciones_sel.empty:
    print("Zona vacía, seleccionando automáticamente una zona válida...")

    minx, miny, maxx, maxy = elevaciones.total_bounds
    ancho = (maxx - minx) * 0.25
    alto  = (maxy - miny) * 0.25

    iniciox = random.uniform(minx, maxx - ancho)
    inicioy = random.uniform(miny, maxy - alto)
    finx    = iniciox + ancho
    finy    = inicioy + alto

    bosques_sel, humedales_sel, elevaciones_sel, zona_gdf = definirzona(
        iniciox, inicioy, finx, finy
    )
    print(f" Nueva zona definida: ({iniciox}, {inicioy}) a ({finx}, {finy})")

# --------------------------
# 4. Preparar grilla de elevaciones
# --------------------------
if elevaciones_sel.empty:
    raise ValueError("No hay puntos de elevación en la zona seleccionada. Prueba otros límites.")

coords = np.array([[geom.centroid.x, geom.centroid.y] for geom in elevaciones_sel.geometry],
                  dtype=np.float64)
z = elevaciones_sel["altura"].astype(float).to_numpy()

nx, ny = 60, 60
xi = np.linspace(coords[:, 0].min(), coords[:, 0].max(), nx)
yi = np.linspace(coords[:, 1].min(), coords[:, 1].max(), ny)
X, Y = np.meshgrid(xi, yi)
Z = griddata((coords[:, 0], coords[:, 1]), z, (X, Y), method="cubic")

# --------------------------
# 5. Factores de expansión
# --------------------------
factor_bosque = {
    "Bosque Amazónico": 0.6,        # Muy húmedo → baja propagación
    "Bosque Andino": 0.9,           # Pendiente favorece propagación, pero baja biomasa
    "Bosque Chaqueño": 1.6,         # Muy inflamable, propagación alta
    "Bosque Chiquitano": 1.2,       # Riesgo intermedio-alto
    "Bosque Secos Interandinos": 1.4, # Muy secos, alta vulnerabilidad
    "Bosque Tucumano-Boliviano": 1.0, # Riesgo medio
    "Bosque de Llanuras Inundables": 0.7, # Baja propagación por humedad
    "Bosque de Pantano": 0.4,       # Mínima propagación
    "Bosque de Yungas": 0.5         # Muy húmedo, casi no arde
}

def calcular_expansion(row):
    bosque = row["tipo_bosqu"]
    humedal = row["tipo"]
    elevacion = row["altura"]

    f_bosque = factor_bosque.get(bosque, 1.0)
    f_humedal = 0.5 if humedal == "Humedal" else 1.0

    if elevacion < 500:
        f_elev = 1.2
    elif elevacion < 1500:
        f_elev = 1.0
    else:
        f_elev = 0.7

    return f_bosque * f_humedal * f_elev

# Crear matriz de factores en la grilla
factores = np.ones_like(Z)

for i in range(Z.shape[0]):
    for j in range(Z.shape[1]):
        punto = gpd.GeoSeries([Point(X[i, j], Y[i, j])], crs=target_crs)
        bosque_match = bosques_sel[bosques_sel.contains(punto[0])]
        humedal_match = humedales_sel[humedales_sel.contains(punto[0])]
        elev = Z[i, j]

        if not bosque_match.empty:
            bosque_tipo = bosque_match.iloc[0]["tipo_bosqu"]
        else:
            bosque_tipo = None

        if not humedal_match.empty:
            humedal_tipo = "Humedal"
        else:
            humedal_tipo = None

        factores[i, j] = calcular_expansion({
            "tipo_bosqu": bosque_tipo,
            "tipo": humedal_tipo,
            "altura": elev
        })

# --------------------------
# 6. Simulación de fuego
# --------------------------
T = np.zeros_like(Z)

# foco inicial en el centro del área
cx, cy = nx // 2, ny // 2
T[cx-2:cx+3, cy-2:cy+3] = 1000  # ignición inicial

# parámetros de propagación
D = 0.4     # difusión
alpha = 0.01
Tign = 200  # temperatura de ignición
dt = 0.1

def laplacian(Z):
    return (np.roll(Z,1,0) + np.roll(Z,-1,0) + np.roll(Z,1,1) + np.roll(Z,-1,1) - 4*Z)

def get_Tem(T):
    S = (T > Tign).astype(float) * 30.0 * factores
    dT = D * laplacian(T) - alpha * T + S
    return T + dT * dt

# --------------------------
# 7. Visualización animada en 3D
# --------------------------
fig = plt.figure(figsize=(9,7))
ax = fig.add_subplot(111, projection="3d")

def animate(frame):
    global T
    T = get_Tem(T)
    ax.clear()
    ax.plot_surface(X, Y, Z, cmap="terrain", alpha=0.5, linewidth=0)
    Thorn = np.clip(T / 1000, 0, 1)
    ax.plot_surface(X, Y, Z + 10, facecolors=plt.cm.hot(Thorn), alpha=0.7)

    # --- Extra: Mostrar bosque y humedal ---
    punto_central = gpd.GeoSeries([Point(X[cx, cy], Y[cx, cy])], crs=target_crs)
    bosque_match = bosques_sel[bosques_sel.contains(punto_central[0])]
    humedal_match = humedales_sel[humedales_sel.contains(punto_central[0])]

    bosque_tipo = bosque_match.iloc[0]["tipo_bosqu"] if not bosque_match.empty else "Ninguno"
    humedal_tipo = "Sí" if not humedal_match.empty else "No"

    ax.text2D(0.05, 0.95,
              f"Bosque: {bosque_tipo}\nHumedal: {humedal_tipo}",
              transform=ax.transAxes,
              fontsize=10, bbox=dict(facecolor="white", alpha=0.7))

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Elevación")

ani = FuncAnimation(fig, animate, frames=120, interval=120, blit=False)
plt.show()


##  SIMULACION DE EXPANSION DE FUEGO PAR.3 -> DATOS CUANTITATIVOS

* Se añade datos cuantitativos para ver como fue dicha expansion del fuego

In [ ]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import box, Point
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from scipy.interpolate import griddata
import random
from collections import Counter

# --------------------------
# 1. Cargar capas
# --------------------------
bosques = gpd.read_file("bosques.gpkg")
humedales = gpd.read_file("humedades.gpkg")
elevaciones = gpd.read_file("elevaciones.gpkg")

# --- Asegurar que trabajamos en CRS métrico ---
if bosques.crs.is_geographic:
    print(f"CRS está en grados ({bosques.crs}), reproyectando a UTM...")
    # Ejemplo: EPSG:32719 (WGS84 / UTM zone 19S) -> Bolivia
    target_crs = "EPSG:32719"
    bosques = bosques.to_crs(target_crs)
    humedales = humedales.to_crs(target_crs)
    elevaciones = elevaciones.to_crs(target_crs)
else:
    target_crs = bosques.crs

# --------------------------
# 2. Función para definir zona de análisis (mínimo 10x10 km)
# --------------------------
def definirzona(iniciox, inicioy, finx, finy, min_size=10000):
    ancho = max(finx - iniciox, min_size)
    alto  = max(finy - inicioy, min_size)

    finx = iniciox + ancho
    finy = inicioy + alto

    zona = box(iniciox, inicioy, finx, finy)
    zona_gdf = gpd.GeoDataFrame(geometry=[zona], crs=target_crs)

    bosques_sel = gpd.clip(bosques, zona_gdf)
    humedales_sel = gpd.clip(humedales, zona_gdf)
    elevaciones_sel = gpd.clip(elevaciones, zona_gdf)

    return bosques_sel, humedales_sel, elevaciones_sel, zona_gdf

# --------------------------
# 3. Seleccionar zona de prueba
# --------------------------
bosques_sel, humedales_sel, elevaciones_sel, zona_gdf = definirzona(
    500000, 8000000, 502000, 8002000
)

if elevaciones_sel.empty:
    print("Zona vacia, seleccionando automaticamente una zona valida...")

    minx, miny, maxx, maxy = elevaciones.total_bounds
    ancho = (maxx - minx) * 0.25
    alto  = (maxy - miny) * 0.25

    iniciox = random.uniform(minx, maxx - ancho)
    inicioy = random.uniform(miny, maxy - alto)
    finx    = iniciox + ancho
    finy    = inicioy + alto

    bosques_sel, humedales_sel, elevaciones_sel, zona_gdf = definirzona(
        iniciox, inicioy, finx, finy
    )
    print(f" Nueva zona definida: ({iniciox}, {inicioy}) a ({finx}, {finy})")

# --------------------------
# 4. Preparar grilla de elevaciones
# --------------------------
if elevaciones_sel.empty:
    raise ValueError("No hay puntos de elevacion en la zona seleccionada. Prueba otros limites.")

coords = np.array([[geom.centroid.x, geom.centroid.y] for geom in elevaciones_sel.geometry],
                  dtype=np.float64)
z = elevaciones_sel["altura"].astype(float).to_numpy()

nx, ny = 60, 60
xi = np.linspace(coords[:, 0].min(), coords[:, 0].max(), nx)
yi = np.linspace(coords[:, 1].min(), coords[:, 1].max(), ny)
X, Y = np.meshgrid(xi, yi)
Z = griddata((coords[:, 0], coords[:, 1]), z, (X, Y), method="cubic")

# --------------------------
# 5. Factores de expansión
# --------------------------
factor_bosque = {
    "Bosque Amazónico": 0.6,
    "Bosque Andino": 0.9,
    "Bosque Chaqueño": 1.6,
    "Bosque Chiquitano": 1.2,
    "Bosque Secos Interandinos": 1.4,
    "Bosque Tucumano-Boliviano": 1.0,
    "Bosque de Llanuras Inundables": 0.7,
    "Bosque de Pantano": 0.4,
    "Bosque de Yungas": 0.5
}

def calcular_expansion(row):
    bosque = row["tipo_bosqu"]
    humedal = row["tipo"]
    elevacion = row["altura"]

    f_bosque = factor_bosque.get(bosque, 1.0)
    f_humedal = 0.5 if humedal == "Humedal" else 1.0

    if elevacion < 500:
        f_elev = 1.2
    elif elevacion < 1500:
        f_elev = 1.0
    else:
        f_elev = 0.7

    return f_bosque * f_humedal * f_elev

# Crear matriz de factores y metadatos
factores = np.ones_like(Z)
bosque_grid = np.full(Z.shape, "Ninguno", dtype=object)
humedal_grid = np.full(Z.shape, "No", dtype=object)

for i in range(Z.shape[0]):
    for j in range(Z.shape[1]):
        punto = gpd.GeoSeries([Point(X[i, j], Y[i, j])], crs=target_crs)
        bosque_match = bosques_sel[bosques_sel.contains(punto[0])]
        humedal_match = humedales_sel[humedales_sel.contains(punto[0])]
        elev = Z[i, j]

        if not bosque_match.empty:
            bosque_tipo = bosque_match.iloc[0]["tipo_bosqu"]
        else:
            bosque_tipo = None

        if not humedal_match.empty:
            humedal_tipo = "Humedal"
        else:
            humedal_tipo = None

        factores[i, j] = calcular_expansion({
            "tipo_bosqu": bosque_tipo,
            "tipo": humedal_tipo,
            "altura": elev
        })
        bosque_grid[i, j] = bosque_tipo if bosque_tipo else "Ninguno"
        humedal_grid[i, j] = "Sí" if humedal_tipo else "No"

# --------------------------
# 6. Simulación de fuego
# --------------------------
T = np.zeros_like(Z)
quemado = np.zeros_like(Z, dtype=bool)

cx, cy = nx // 2, ny // 2
T[cx-2:cx+3, cy-2:cy+3] = 1000

D = 0.4
alpha = 0.01
Tign = 200
dt = 0.1

def laplacian(Z):
    return (np.roll(Z,1,0) + np.roll(Z,-1,0) + np.roll(Z,1,1) + np.roll(Z,-1,1) - 4*Z)

def get_Tem(T):
    global quemado
    S = (T > Tign).astype(float) * 30.0 * factores
    dT = D * laplacian(T) - alpha * T + S
    Tn = T + dT * dt
    quemado |= (Tn > Tign)
    return Tn

# --------------------------
# 7. Visualización animada
# --------------------------
fig = plt.figure(figsize=(9,7))
ax = fig.add_subplot(111, projection="3d")

def animate(frame):
    global T
    T = get_Tem(T)
    ax.clear()
    ax.plot_surface(X, Y, Z, cmap="terrain", alpha=0.5, linewidth=0)
    Thorn = np.clip(T / 1000, 0, 1)
    ax.plot_surface(X, Y, Z + 10, facecolors=plt.cm.hot(Thorn), alpha=0.7)

    punto_central = gpd.GeoSeries([Point(X[cx, cy], Y[cx, cy])], crs=target_crs)
    bosque_match = bosques_sel[bosques_sel.contains(punto_central[0])]
    humedal_match = humedales_sel[humedales_sel.contains(punto_central[0])]

    bosque_tipo = bosque_match.iloc[0]["tipo_bosqu"] if not bosque_match.empty else "Ninguno"
    humedal_tipo = "Sí" if not humedal_match.empty else "No"

    ax.text2D(0.05, 0.95,
              f"Bosque: {bosque_tipo}\nHumedal: {humedal_tipo}",
              transform=ax.transAxes,
              fontsize=10, bbox=dict(facecolor="white", alpha=0.7))

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Elevacion")

ani = FuncAnimation(fig, animate, frames=120, interval=120, blit=False)
plt.show()

# --------------------------
# 8. Resultados cuantitativos
# --------------------------
tiempo_total = 120 * dt

# --- Calculo del área en km² ---
dx = (X.max() - X.min()) / nx
dy = (Y.max() - Y.min()) / ny
area_celda_km2 = (dx * dy) / 1e6
area_quemada_km2 = quemado.sum() * area_celda_km2

velocidad_promedio = area_quemada_km2 / tiempo_total

# bosque más afectado
bosques_afectados = Counter(bosque_grid[quemado])
bosque_mas_afectado = bosques_afectados.most_common(1)[0] if bosques_afectados else ("Ninguno", 0)

# porcentaje en humedales
total_quemado = quemado.sum()
humedal_quemado = np.sum((quemado) & (humedal_grid == "Sí"))
pct_humedal = (humedal_quemado / total_quemado * 100) if total_quemado > 0 else 0

print("\n Resultados de la simulacion:")
print(f" Tiempo total simulado: {tiempo_total:.2f} unidades de tiempo")
print(f" Area total quemada: {area_quemada_km2:.4f} km^2")
print(f" Velocidad promedio de propagacion: {velocidad_promedio:.4f} km^2/unidad de tiempo")
print(f" Bosque mas afectado: {bosque_mas_afectado[0]} ({bosque_mas_afectado[1]} celdas)")
print(f" % de propagacion en humedales: {pct_humedal:.2f}%")


## SIMULACION DE EXPANSION DE FUEGO PAR.4 -> Viento Agregado

* Se añade el parametro de viento y la seleccion de coordenas es aleatoria conforme a todo el mapa

In [ ]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import box, Point
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from scipy.interpolate import griddata
import random
from collections import Counter

# --------------------------
# 1. Cargar capas
# --------------------------
bosques = gpd.read_file("bosques.gpkg")
humedales = gpd.read_file("humedades.gpkg")
elevaciones = gpd.read_file("elevaciones.gpkg")

# --- Asegurar CRS métrico ---
if bosques.crs.is_geographic:
    target_crs = "EPSG:32719"  # WGS84 / UTM zone 19S (Bolivia)
    bosques = bosques.to_crs(target_crs)
    humedales = humedales.to_crs(target_crs)
    elevaciones = elevaciones.to_crs(target_crs)
else:
    target_crs = bosques.crs

# --------------------------
# 2. Función para definir zona de análisis
# --------------------------
def definirzona(iniciox=None, inicioy=None, finx=None, finy=None, min_size=10000):
    """Si no se pasan coordenadas, selecciona zona aleatoria dentro de Bolivia"""
    if iniciox is None or inicioy is None or finx is None or finy is None:
        minx, miny, maxx, maxy = elevaciones.total_bounds
        ancho = (maxx - minx) * 0.25
        alto  = (maxy - miny) * 0.25
        iniciox = random.uniform(minx, maxx - ancho)
        inicioy = random.uniform(miny, maxy - alto)
        finx = iniciox + ancho
        finy = inicioy + alto

    # Aseguramos tamaño mínimo
    ancho = max(finx - iniciox, min_size)
    alto  = max(finy - inicioy, min_size)
    finx = iniciox + ancho
    finy = inicioy + alto

    zona = box(iniciox, inicioy, finx, finy)
    zona_gdf = gpd.GeoDataFrame(geometry=[zona], crs=target_crs)

    bosques_sel = gpd.clip(bosques, zona_gdf)
    humedales_sel = gpd.clip(humedales, zona_gdf)
    elevaciones_sel = gpd.clip(elevaciones, zona_gdf)

    return bosques_sel, humedales_sel, elevaciones_sel, zona_gdf

# --------------------------
# 3. Seleccionar zona de prueba (puede ser todo Bolivia)
# --------------------------
bosques_sel, humedales_sel, elevaciones_sel, zona_gdf = definirzona()

if elevaciones_sel.empty:
    raise ValueError("Zona seleccionada no tiene puntos de elevacion. Reintente.")

# --------------------------
# 4. Preparar grilla de elevaciones
# --------------------------
coords = np.array([[geom.centroid.x, geom.centroid.y] for geom in elevaciones_sel.geometry], dtype=np.float64)
z = elevaciones_sel["altura"].astype(float).to_numpy()

nx, ny = 60, 60
xi = np.linspace(coords[:, 0].min(), coords[:, 0].max(), nx)
yi = np.linspace(coords[:, 1].min(), coords[:, 1].max(), ny)
X, Y = np.meshgrid(xi, yi)
Z = griddata((coords[:, 0], coords[:, 1]), z, (X, Y), method="cubic")

# --------------------------
# 5. Factores de expansión
# --------------------------
factor_bosque = {
    "Bosque Amazónico": 0.6,
    "Bosque Andino": 0.9,
    "Bosque Chaqueño": 1.6,
    "Bosque Chiquitano": 1.2,
    "Bosque Secos Interandinos": 1.4,
    "Bosque Tucumano-Boliviano": 1.0,
    "Bosque de Llanuras Inundables": 0.7,
    "Bosque de Pantano": 0.4,
    "Bosque de Yungas": 0.5
}

def calcular_expansion(row):
    bosque = row["tipo_bosqu"]
    humedal = row["tipo"]
    elevacion = row["altura"]

    f_bosque = factor_bosque.get(bosque, 1.0)
    f_humedal = 0.5 if humedal == "Humedal" else 1.0

    if elevacion < 500:
        f_elev = 1.2
    elif elevacion < 1500:
        f_elev = 1.0
    else:
        f_elev = 0.7

    return f_bosque * f_humedal * f_elev

factores = np.ones_like(Z)
bosque_grid = np.full(Z.shape, "Ninguno", dtype=object)
humedal_grid = np.full(Z.shape, "No", dtype=object)

for i in range(Z.shape[0]):
    for j in range(Z.shape[1]):
        punto = gpd.GeoSeries([Point(X[i, j], Y[i, j])], crs=target_crs)
        bosque_match = bosques_sel[bosques_sel.contains(punto[0])]
        humedal_match = humedales_sel[humedales_sel.contains(punto[0])]
        elev = Z[i, j]

        bosque_tipo = bosque_match.iloc[0]["tipo_bosqu"] if not bosque_match.empty else None
        humedal_tipo = "Humedal" if not humedal_match.empty else None

        factores[i, j] = calcular_expansion({
            "tipo_bosqu": bosque_tipo,
            "tipo": humedal_tipo,
            "altura": elev
        })
        bosque_grid[i, j] = bosque_tipo if bosque_tipo else "Ninguno"
        humedal_grid[i, j] = "Sí" if humedal_tipo else "No"

# --------------------------
# 6. Simulación de fuego con viento
# --------------------------
T = np.zeros_like(Z)
quemado = np.zeros_like(Z, dtype=bool)
cx, cy = nx // 2, ny // 2
T[cx-2:cx+3, cy-2:cy+3] = 1000

D = 0.4
alpha = 0.01
Tign = 200
dt = 0.1

# Parámetros de viento (vx, vy) en unidades de celdas por dt
vx, vy = 1.0, 0.5  # Ejemplo: viento hacia X positivo y Y positivo
wind_factor = 1.5   # Incremento de propagación en la dirección del viento

def laplacian(Z):
    return (np.roll(Z,1,0) + np.roll(Z,-1,0) + np.roll(Z,1,1) + np.roll(Z,-1,1) - 4*Z)

def wind_effect(T):
    """Aumenta T en la direccion del viento y disminuye en la opuesta"""
    shifted_x = np.roll(T, -int(vx), axis=1)
    shifted_y = np.roll(T, -int(vy), axis=0)
    return 0.5*(shifted_x + shifted_y) * wind_factor

def get_Tem(T):
    global quemado
    S = (T > Tign).astype(float) * 30.0 * factores
    dT = D * laplacian(T) - alpha * T + S
    dT += wind_effect(T) - T  # efecto del viento
    Tn = T + dT * dt
    quemado |= (Tn > Tign)
    return Tn

# --------------------------
# 7. Visualización animada
# --------------------------
fig = plt.figure(figsize=(9,7))
ax = fig.add_subplot(111, projection="3d")

def animate(frame):
    global T
    T = get_Tem(T)
    ax.clear()
    ax.plot_surface(X, Y, Z, cmap="terrain", alpha=0.5, linewidth=0)
    Thorn = np.clip(T / 1000, 0, 1)
    ax.plot_surface(X, Y, Z + 10, facecolors=plt.cm.hot(Thorn), alpha=0.7)

    punto_central = gpd.GeoSeries([Point(X[cx, cy], Y[cx, cy])], crs=target_crs)
    bosque_match = bosques_sel[bosques_sel.contains(punto_central[0])]
    humedal_match = humedales_sel[humedales_sel.contains(punto_central[0])]
    bosque_tipo = bosque_match.iloc[0]["tipo_bosqu"] if not bosque_match.empty else "Ninguno"
    humedal_tipo = "Sí" if not humedal_match.empty else "No"

    ax.text2D(0.05, 0.95,
              f"Bosque: {bosque_tipo}\nHumedal: {humedal_tipo}",
              transform=ax.transAxes,
              fontsize=10, bbox=dict(facecolor="white", alpha=0.7))

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Elevacion")

ani = FuncAnimation(fig, animate, frames=120, interval=120, blit=False)
plt.show()

# --------------------------
# 8. Resultados cuantitativos
# --------------------------
tiempo_total = 120 * dt
dx = (X.max() - X.min()) / nx
dy = (Y.max() - Y.min()) / ny
area_celda_km2 = (dx * dy) / 1e6
area_quemada_km2 = quemado.sum() * area_celda_km2
velocidad_promedio = area_quemada_km2 / tiempo_total

bosques_afectados = Counter(bosque_grid[quemado])
bosque_mas_afectado = bosques_afectados.most_common(1)[0] if bosques_afectados else ("Ninguno", 0)
total_quemado = quemado.sum()
humedal_quemado = np.sum((quemado) & (humedal_grid == "Sí"))
pct_humedal = (humedal_quemado / total_quemado * 100) if total_quemado > 0 else 0

print("\n Resultados de la simulacion:")
print(f" Tiempo total simulado: {tiempo_total:.2f} unidades de tiempo")
print(f" Area total quemada: {area_quemada_km2:.4f} km^2")
print(f" Velocidad promedio de propagacion: {velocidad_promedio:.4f} km^2/unidad de tiempo")
print(f" Bosque mas afectado: {bosque_mas_afectado[0]} ({bosque_mas_afectado[1]} celdas)")
print(f" % de propagacion en humedales: {pct_humedal:.2f}%")
